# MeCrab Python API - Basic Usage

**Date:** 2026-01-02  
**Author:** COOLJAPAN OU (Team KitaSan)  
**License:** MIT OR Apache-2.0

This notebook demonstrates the basic usage of MeCrab Python bindings.

## Installation

First, install MeCrab Python package:

```bash
# From source (requires Rust toolchain)
cd /path/to/mecrab
maturin develop --features python,parallel

# Or from PyPI (when published)
pip install mecrab
```

Make sure you have IPADIC dictionary installed:

```bash
# Ubuntu/Debian
sudo apt install mecab-ipadic-utf8

# macOS
brew install mecab-ipadic
```

## 1. Import and Version Check

In [ ]:
import mecrab

print(f"MeCrab version: {mecrab.version()}")

## 2. Initialize MeCrab

Create a MeCrab instance with default dictionary path:

In [ ]:
# Use default dictionary (auto-detect IPADIC location)
m = mecrab.MeCrab()

# Or specify dictionary path explicitly
# m = mecrab.MeCrab(dicdir="/var/lib/mecab/dic/ipadic-utf8")

## 3. Basic Parsing

In [ ]:
# Parse a sentence
text = "すもももももももものうち"
result = m.parse(text)
print(result)

## 4. Wakati (Space-separated) Output

Useful for word frequency analysis:

In [ ]:
text = "私は学生です"
words = m.wakati(text)
print(words)
print("\nTokens:", words.split())

## 5. Parse to List (Structured Output)

In [ ]:
text = "東京に行く"
morphemes = m.parse_to_list(text)

for surface, feature in morphemes:
    # Feature format: POS,POS1,POS2,POS3,Inflection,Conjugation,Base,Reading,Pronunciation
    parts = feature.split(',')
    pos = parts[0]
    reading = parts[7] if len(parts) > 7 else ""
    print(f"{surface:8s} {pos:6s} {reading}")

## 6. Batch Processing

Process multiple texts in parallel (requires `parallel` feature):

In [ ]:
texts = [
    "今日は良い天気です",
    "明日は雨が降るでしょう",
    "週末は買い物に行きます"
]

# Batch parse (parallel processing)
results = m.parse_batch(texts)

for i, result in enumerate(results, 1):
    print(f"\n--- Text {i} ---")
    print(result)

In [ ]:
# Batch wakati (space-separated)
wakati_results = m.wakati_batch(texts)

for i, words in enumerate(wakati_results, 1):
    print(f"Text {i}: {words}")

## 7. Adding Custom Words (Overlay Dictionary)

Add new words at runtime without modifying the dictionary files:

In [ ]:
# Before adding custom word
text = "ChatGPTは便利です"
print("Before:")
print(m.parse(text))

# Add custom word: surface, reading, pronunciation, cost
# Lower cost = more preferred (typical: 5000-8000)
m.add_word("ChatGPT", "チャットジーピーティー", "チャットジーピーティー", 5000)

print("\nAfter adding custom word:")
print(m.parse(text))

print(f"\nOverlay dictionary size: {m.overlay_size()} words")

## 8. Practical Example: Word Frequency Counter

In [ ]:
from collections import Counter

# Sample text corpus
corpus = """
東京は日本の首都です。
東京には多くの人が住んでいます。
東京タワーは有名な観光地です。
日本の文化は素晴らしいです。
"""

# Tokenize and count
all_words = []
for line in corpus.strip().split('\n'):
    if line:
        words = m.wakati(line).split()
        all_words.extend(words)

# Count frequencies
freq = Counter(all_words)

print("Top 10 most frequent words:")
for word, count in freq.most_common(10):
    print(f"{word:10s} {count:3d}")

## 9. Practical Example: Extract Nouns

In [ ]:
text = "東京の美しい公園で友達と散歩をしました"

morphemes = m.parse_to_list(text)
nouns = [surface for surface, feature in morphemes if feature.startswith('名詞')]

print(f"Input: {text}")
print(f"Nouns: {', '.join(nouns)}")

## 10. Performance Benchmark

Compare single vs batch processing:

In [ ]:
import time

# Generate test data
test_texts = ["今日は良い天気です"] * 1000

# Single processing
start = time.time()
for text in test_texts:
    _ = m.parse(text)
single_time = time.time() - start

# Batch processing
start = time.time()
_ = m.parse_batch(test_texts)
batch_time = time.time() - start

print(f"Single processing: {single_time:.3f} sec")
print(f"Batch processing:  {batch_time:.3f} sec")
print(f"Speedup: {single_time/batch_time:.2f}x")

## Next Steps

- **Advanced Tutorial:** IPA pronunciation, word embeddings, semantic URIs
- **Word2Vec Training:** See `02_word2vec_training.ipynb`
- **Custom Applications:** Build text classification, NER, etc.

## Documentation

- [MeCrab GitHub](https://github.com/kitasan/mecrab)
- [Rust API docs](https://docs.rs/mecrab)
- [CLI documentation](https://github.com/kitasan/mecrab/tree/main/kizame)

---

**Copyright 2026 COOLJAPAN OU (Team KitaSan)**  
**License:** MIT OR Apache-2.0